# Galaxy Zoo Data Exploration

Use this notebook to download the Galaxy Zoo Kaggle competition data, extract the training images and labels into the project, and inspect them before building degradation, reconstruction, or model-training code.

Expected local layout after running the setup cells:

```text
data/
  raw/
    images_training_rev1/   # extracted training images
    training_solutions_rev1.csv
  processed/                # cleaned/resized images later
  degraded/                 # simulated blur/noise/resolution loss later
  reconstructed/            # reconstructed images later
```


In [ ]:
from pathlib import Path
from zipfile import ZipFile

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

for directory in [RAW_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, RAW_DIR


## Download and Extract Data

This uses KaggleHub to download or locate the Kaggle competition archives, then extracts the files used in this notebook into `data/raw/`. KaggleHub reuses its cache on later runs.


In [ ]:
KAGGLE_COMPETITION = "galaxy-zoo-the-galaxy-challenge"
ARCHIVES_TO_EXTRACT = [
    "training_solutions_rev1.zip",
    "images_training_rev1.zip",
]

kaggle_cache_dir = Path(kagglehub.competition_download(KAGGLE_COMPETITION))
print(f"Kaggle archive cache: {kaggle_cache_dir}")

for archive_name in ARCHIVES_TO_EXTRACT:
    archive_path = kaggle_cache_dir / archive_name
    if not archive_path.exists():
        raise FileNotFoundError(f"Expected archive not found: {archive_path}")

    with ZipFile(archive_path) as archive:
        members = [member for member in archive.namelist() if not member.endswith("/")]
        missing_members = [member for member in members if not (RAW_DIR / member).exists()]
        if missing_members:
            archive.extractall(RAW_DIR, members=missing_members)
            print(f"Extracted {len(missing_members):,} files from {archive_name}")
        else:
            print(f"Already extracted: {archive_name}")


## Discover Local Files

After the setup cells run, the notebook reads the extracted images and labels from `data/raw/`.


In [ ]:
all_files = sorted(path for path in RAW_DIR.rglob("*") if path.is_file())

print(f"Found {len(all_files):,} extracted files in {RAW_DIR}")
for path in all_files[:20]:
    print(path.relative_to(PROJECT_ROOT))


In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
table_extensions = {".csv", ".parquet", ".tsv"}

image_paths = [path for path in all_files if path.suffix.lower() in image_extensions]
table_paths = [path for path in all_files if path.suffix.lower() in table_extensions]

training_image_paths = [path for path in image_paths if "images_training_rev1" in path.parts]
label_candidates = [path for path in table_paths if path.name == "training_solutions_rev1.csv"]

print(f"Images: {len(image_paths):,}")
print(f"Training images: {len(training_image_paths):,}")
print(f"Tables: {len(table_paths):,}")

for path in table_paths[:10]:
    print(path.relative_to(PROJECT_ROOT))


## Inspect Label Tables

The Galaxy Zoo competition labels are expected at `data/raw/training_solutions_rev1.csv` after extraction.


In [ ]:
if label_candidates:
    label_path = label_candidates[0]
    if label_path.suffix.lower() == ".parquet":
        labels = pd.read_parquet(label_path)
    elif label_path.suffix.lower() == ".tsv":
        labels = pd.read_csv(label_path, sep="\t")
    else:
        labels = pd.read_csv(label_path)

    print(label_path.relative_to(PROJECT_ROOT))
    display(labels.head())
    display(labels.info())
else:
    labels = None
    print("No training label table found. Run the download/extract cell above first.")


In [ ]:
if labels is not None:
    display(labels.describe(include="all").T.head(30))

## Preview Images

In [ ]:
sample_paths = training_image_paths[:12]

if sample_paths:
    n_cols = 4
    n_rows = int(np.ceil(len(sample_paths) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, path in zip(axes, sample_paths):
        image = Image.open(path).convert("RGB")
        ax.imshow(image)
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")

    for ax in axes[len(sample_paths):]:
        ax.axis("off")

    plt.tight_layout()
else:
    print("No training images found. Run the download/extract cell above first.")


## Check Image Sizes

This helps decide the resize/crop size for the later processed dataset.

In [ ]:
size_records = []

for path in training_image_paths[:1000]:
    with Image.open(path) as image:
        width, height = image.size
    size_records.append({"path": str(path.relative_to(PROJECT_ROOT)), "width": width, "height": height})

sizes = pd.DataFrame(size_records)
if not sizes.empty:
    display(sizes.head())
    display(sizes[["width", "height"]].describe())
else:
    print("No training image sizes to inspect. Run the download/extract cell above first.")
